# Bygg komplett FPL-datasett for alle sesonger

Notebooken leser hver sesongs `gws/merged_gw.csv`, slår sesongene sammen, legger til motstandernavn og beregner lagstatistikk. Originalfilene endres ikke.

Resultatet lagres som `data-source/data/cleaned_merged_seasons_team_aggregated_expanded.csv`.

In [1]:
%pip install -q pandas


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Finn data og sesonger

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)

candidates = [
    Path.cwd() / "data-source" / "data",
    Path.cwd().parent / "data-source" / "data",
]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Fant ikke data-source/data")

seasons = sorted(
    path.name
    for path in DATA_DIR.iterdir()
    if path.is_dir()
    and path.name[:4].isdigit()
    and (path / "gws" / "merged_gw.csv").exists()
)

print("Datamappe:", DATA_DIR)
print("Sesonger som skal inkluderes:", seasons)

Datamappe: /Users/henrik/Documents/fplmodell/data-source/data
Sesonger som skal inkluderes: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']


## 2. Last inn og slå sammen gameweek-data

Kolonnene har endret seg mellom sesongene. `concat(..., sort=False)` beholder unionen av alle kolonner og fyller manglende historiske felt med `NaN`.

In [3]:
frames = []
load_report = []

for season in seasons:
    path = DATA_DIR / season / "gws" / "merged_gw.csv"
    try:
        season_data = pd.read_csv(path)
    except UnicodeDecodeError:
        season_data = pd.read_csv(path, encoding="latin-1")

    season_data["season_x"] = season
    if "GW" not in season_data.columns and "round" in season_data.columns:
        season_data["GW"] = season_data["round"]

    frames.append(season_data)
    load_report.append({
        "season": season,
        "rows": len(season_data),
        "columns": len(season_data.columns),
        "gameweeks": season_data["GW"].nunique(),
    })

all_seasons = pd.concat(frames, ignore_index=True, sort=False)
all_seasons["kickoff_time"] = pd.to_datetime(
    all_seasons["kickoff_time"], errors="coerce", utc=True
)

# Kildedataene har enkelte identiske rader og noen utsatte kamper registrert
# både på opprinnelig og ny dato. Behold siste rad per spiller og fixture.
player_fixture_key = ["season_x", "element", "fixture"]
rows_before_deduplication = len(all_seasons)
all_seasons = (
    all_seasons.sort_values(player_fixture_key + ["kickoff_time"], na_position="first")
    .drop_duplicates(player_fixture_key, keep="last")
    .reset_index(drop=True)
)
print(f"Fjernet {rows_before_deduplication - len(all_seasons)} duplikatrader")

# De eldste sesongene mangler spillerens lag. I hver kamp er spillerens
# lag den andre unike opponent_team-ID-en som finnes i samme fixture.
participants = all_seasons[["season_x", "fixture", "opponent_team"]].drop_duplicates()
own_team_lookup = participants.merge(
    participants, on=["season_x", "fixture"], suffixes=("", "_other")
)
own_team_lookup = own_team_lookup.loc[
    own_team_lookup["opponent_team"].ne(own_team_lookup["opponent_team_other"]),
    ["season_x", "fixture", "opponent_team", "opponent_team_other"],
].rename(columns={"opponent_team_other": "derived_team_id"})
own_team_lookup = own_team_lookup.drop_duplicates(
    ["season_x", "fixture", "opponent_team"]
)
all_seasons = all_seasons.merge(
    own_team_lookup, on=["season_x", "fixture", "opponent_team"], how="left",
    validate="many_to_one",
)

# Lag en sesongspesifikk ID->navn-tabell fra teams.csv og masterlisten.
own_name_frames = []
for season in seasons:
    teams_path = DATA_DIR / season / "teams.csv"
    if teams_path.exists():
        names = pd.read_csv(teams_path, usecols=["id", "name"]).rename(
            columns={"id": "derived_team_id", "name": "derived_team_name"}
        )
        names["season_x"] = season
        own_name_frames.append(names)
master_path = DATA_DIR / "master_team_list.csv"
if master_path.exists():
    names = pd.read_csv(master_path)[["season", "team", "team_name"]].rename(
        columns={"season": "season_x", "team": "derived_team_id", "team_name": "derived_team_name"}
    )
    own_name_frames.append(names)
own_names = pd.concat(own_name_frames, ignore_index=True).drop_duplicates(
    ["season_x", "derived_team_id"]
)
all_seasons = all_seasons.merge(
    own_names, on=["season_x", "derived_team_id"], how="left", validate="many_to_one"
)
if "team" not in all_seasons.columns:
    all_seasons["team"] = all_seasons["derived_team_name"]
else:
    all_seasons["team"] = all_seasons["team"].fillna(all_seasons["derived_team_name"])
all_seasons = all_seasons.drop(columns=["derived_team_name"])
if "position" in all_seasons.columns:
    all_seasons["position"] = all_seasons.groupby("name")["position"].transform(
        lambda values: values.ffill().bfill()
    )

display(pd.DataFrame(load_report))
print(f"Totalt: {len(all_seasons):,} rader x {len(all_seasons.columns)} kolonner")

Fjernet 69 duplikatrader


,season,rows,columns,gameweeks
0,2016-17,23679,57,38
1,2017-18,22467,57,38
2,2018-19,21790,57,38
3,2019-20,22560,34,38
4,2020-21,24365,37,38
5,2021-22,25447,37,38
6,2022-23,26505,42,37
7,2023-24,29725,42,38
8,2024-25,27605,50,38
9,2025-26,29757,47,38


Totalt: 254,441 rader x 75 kolonner


## 3. Legg til motstandernavn

Lag-ID-er er sesongspesifikke, så koblingen gjøres med både sesong og `opponent_team`.

In [4]:
team_frames = []
for season in seasons:
    teams_path = DATA_DIR / season / "teams.csv"
    if not teams_path.exists():
        continue
    teams = pd.read_csv(teams_path, usecols=["id", "name"])
    teams = teams.rename(columns={"id": "opponent_team", "name": "opp_team_name"})
    teams["season_x"] = season
    team_frames.append(teams)

all_seasons = all_seasons.drop(columns=["opp_team_name"], errors="ignore")
if team_frames:
    opponent_lookup = pd.concat(team_frames, ignore_index=True).drop_duplicates(
        ["season_x", "opponent_team"]
    )
    all_seasons = all_seasons.merge(
        opponent_lookup, on=["season_x", "opponent_team"], how="left"
    )

# Reservekobling for eldre sesonger uten teams.csv.
master_path = DATA_DIR / "master_team_list.csv"
if master_path.exists():
    master = pd.read_csv(master_path)
    master_lookup = master[["season", "team", "team_name"]].rename(
        columns={"season": "season_x", "team": "opponent_team", "team_name": "fallback_name"}
    )
    master_lookup = master_lookup.drop_duplicates(["season_x", "opponent_team"])
    all_seasons = all_seasons.merge(
        master_lookup, on=["season_x", "opponent_team"], how="left"
    )
    all_seasons["opp_team_name"] = all_seasons["opp_team_name"].fillna(
        all_seasons["fallback_name"]
    )
    all_seasons = all_seasons.drop(columns="fallback_name")

print("Andel med motstandernavn:", all_seasons["opp_team_name"].notna().mean().round(3))

Andel med motstandernavn: 1.0


## 4. Beregn lagstatistikk

Vi lager først én rad per lag og kamp. Bruk av `fixture` gjør at double gameweeks behandles riktig.

In [5]:
required = {"season_x", "team", "fixture", "GW", "kickoff_time", "was_home", "team_h_score", "team_a_score"}
missing = required.difference(all_seasons.columns)
if missing:
    raise ValueError(f"Mangler kolonner for lagaggregering: {sorted(missing)}")

team_matches = (
    all_seasons[list(required)]
    .drop_duplicates(["season_x", "team", "fixture"])
    .copy()
)

for column in ["team_h_score", "team_a_score"]:
    team_matches[column] = pd.to_numeric(team_matches[column], errors="coerce")

team_matches["team_goals_scored_match"] = team_matches["team_h_score"].where(
    team_matches["was_home"], team_matches["team_a_score"]
)
team_matches["team_goals_conceded_match"] = team_matches["team_a_score"].where(
    team_matches["was_home"], team_matches["team_h_score"]
)

played = team_matches["team_goals_scored_match"].notna() & team_matches["team_goals_conceded_match"].notna()
won = team_matches["team_goals_scored_match"] > team_matches["team_goals_conceded_match"]
drawn = team_matches["team_goals_scored_match"] == team_matches["team_goals_conceded_match"]
# Bruk float fra starten. pd.NA ville gitt object-dtype, som ikke støtter cumsum.
team_matches["points_match"] = float("nan")
team_matches.loc[played, "points_match"] = 0.0
team_matches.loc[played & drawn, "points_match"] = 1.0
team_matches.loc[played & won, "points_match"] = 3.0
team_matches["points_match"] = pd.to_numeric(
    team_matches["points_match"], errors="coerce"
)

team_matches = team_matches.sort_values(
    ["season_x", "team", "kickoff_time", "fixture"]
).reset_index(drop=True)

group_keys = [team_matches["season_x"], team_matches["team"]]
team_matches["points"] = team_matches["points_match"].fillna(0.0).groupby(group_keys).cumsum()
team_matches["team_goals_scored"] = pd.to_numeric(
    team_matches["team_goals_scored_match"], errors="coerce"
).fillna(0.0).groupby(group_keys).cumsum()
team_matches["team_goals_conceded"] = pd.to_numeric(
    team_matches["team_goals_conceded_match"], errors="coerce"
).fillna(0.0).groupby(group_keys).cumsum()
team_matches["team_goals_diff"] = (
    team_matches["team_goals_scored"] - team_matches["team_goals_conceded"]
)

display(team_matches.head())

,fixture,team,was_home,GW,team_a_score,team_h_score,season_x,kickoff_time,team_goals_scored_match,team_goals_conceded_match,points_match,points,team_goals_scored,team_goals_conceded,team_goals_diff
0,8,Arsenal,True,1,4.0,3.0,2016-17,2016-08-14 15:00:00+00:00,3.0,4.0,0.0,0.0,3.0,4.0,-1.0
1,13,Arsenal,False,2,0.0,0.0,2016-17,2016-08-20 16:30:00+00:00,0.0,0.0,1.0,1.0,3.0,4.0,-1.0
2,28,Arsenal,False,3,3.0,1.0,2016-17,2016-08-27 14:00:00+00:00,3.0,1.0,3.0,4.0,6.0,5.0,1.0
3,31,Arsenal,True,4,1.0,2.0,2016-17,2016-09-10 14:00:00+00:00,2.0,1.0,3.0,7.0,8.0,6.0,2.0
4,43,Arsenal,False,5,4.0,1.0,2016-17,2016-09-17 14:00:00+00:00,4.0,1.0,3.0,10.0,12.0,7.0,5.0


## 5. Lag datalekkasjesikre før-kamp-variabler

Vanlige kumulative kolonner inkluderer den aktuelle kampen. `*_before_match` inneholder bare lagets resultater før kampen og er bedre egnet som modellvariabler.

In [6]:
for source, target in {
    "points": "points_before_match",
    "team_goals_scored": "team_goals_scored_before_match",
    "team_goals_conceded": "team_goals_conceded_before_match",
    "team_goals_diff": "team_goals_diff_before_match",
}.items():
    team_matches[target] = (
        team_matches.groupby(["season_x", "team"], sort=False)[source]
        .shift(1)
        .fillna(0)
    )

display(
    team_matches[[
        "season_x", "team", "GW", "fixture", "points_match",
        "points", "points_before_match", "team_goals_diff_before_match"
    ]].head(20)
)

,season_x,team,GW,fixture,points_match,points,points_before_match,team_goals_diff_before_match
0,2016-17,Arsenal,1,8,0.0,0.0,0.0,0.0
1,2016-17,Arsenal,2,13,1.0,1.0,0.0,-1.0
2,2016-17,Arsenal,3,28,3.0,4.0,1.0,-1.0
3,2016-17,Arsenal,4,31,3.0,7.0,4.0,1.0
4,2016-17,Arsenal,5,43,3.0,10.0,7.0,2.0
5,2016-17,Arsenal,6,51,3.0,13.0,10.0,5.0
6,2016-17,Arsenal,7,61,3.0,16.0,13.0,8.0
7,2016-17,Arsenal,8,71,3.0,19.0,16.0,9.0
8,2016-17,Arsenal,9,81,1.0,20.0,19.0,10.0
9,2016-17,Arsenal,10,97,3.0,23.0,20.0,10.0


## 6. Koble lagdata tilbake til spillerradene

In [7]:
aggregate_columns = [
    "points", "team_goals_scored", "team_goals_conceded", "team_goals_diff",
    "points_before_match", "team_goals_scored_before_match",
    "team_goals_conceded_before_match", "team_goals_diff_before_match",
]
all_seasons = all_seasons.drop(columns=aggregate_columns, errors="ignore")

team_lookup = team_matches[["season_x", "team", "fixture"] + aggregate_columns]
expanded = all_seasons.merge(
    team_lookup, on=["season_x", "team", "fixture"], how="left", validate="many_to_one"
)

if "value" in expanded.columns:
    expanded["price_m"] = pd.to_numeric(expanded["value"], errors="coerce") / 10

expanded = expanded.sort_values(
    ["season_x", "GW", "kickoff_time", "team", "name"]
).reset_index(drop=True)

print(f"Ferdig datasett: {len(expanded):,} rader x {len(expanded.columns)} kolonner")
display(expanded.head())

Ferdig datasett: 254,441 rader x 85 kolonner


,name,assists,attempted_passes,big_chances_created,big_chances_missed,bonus,bps,clean_sheets,clearances_blocks_interceptions,completed_passes,creativity,dribbles,ea_index,element,errors_leading_to_goal,errors_leading_to_goal_attempt,fixture,fouls,goals_conceded,goals_scored,ict_index,id,influence,key_passes,kickoff_time,kickoff_time_formatted,loaned_in,loaned_out,minutes,offside,open_play_crosses,opponent_team,own_goals,penalties_conceded,penalties_missed,penalties_saved,recoveries,red_cards,round,saves,selected,tackled,tackles,target_missed,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,winning_goals,yellow_cards,GW,season_x,position,team,xP,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,starts,mng_clean_sheets,mng_draw,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win,modified,defensive_contribution,derived_team_id,opp_team_name,points,team_goals_scored,team_goals_conceded,team_goals_diff,points_before_match,team_goals_scored_before_match,team_goals_conceded_before_match,team_goals_diff_before_match,price_m
0,Abel_Hernández,1,15.0,0.0,0.0,0,10,0,0.0,10.0,12.2,0.0,0.0,163,0.0,0.0,4,1.0,1,0,5.7,163.0,14.4,1.0,2016-08-13 11:30:00+00:00,13 Aug 12:30,0.0,0.0,90,1.0,0.0,8,0,0.0,0,0,1.0,0,1,0,26039,2.0,0.0,2.0,1.0,2.0,30.0,5,0,0,0,60,True,0.0,0,1,2016-17,NaN,Hull,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,Leicester,3.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,6.0
1,Adama_Diomande,0,28.0,0.0,0.0,2,29,0,3.0,20.0,16.8,3.0,0.0,164,0.0,1.0,4,1.0,1,1,10.7,164.0,45.2,1.0,2016-08-13 11:30:00+00:00,13 Aug 12:30,0.0,0.0,90,0.0,0.0,8,0,0.0,0,0,6.0,0,1,0,38151,6.0,0.0,0.0,1.0,2.0,45.0,8,0,0,0,45,True,0.0,0,1,2016-17,NaN,Hull,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,Leicester,3.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,4.5
2,Ahmed_Elmohamady,0,61.0,0.0,0.0,0,14,0,5.0,38.0,11.4,0.0,0.0,158,0.0,0.0,4,1.0,1,0,2.5,158.0,9.2,1.0,2016-08-13 11:30:00+00:00,13 Aug 12:30,0.0,0.0,90,0.0,0.0,8,0,0.0,0,0,12.0,0,1,0,14379,0.0,1.0,0.0,1.0,2.0,4.0,2,0,0,0,50,True,0.0,0,1,2016-17,NaN,Hull,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,Leicester,3.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,5.0
3,Alex_Bruce,0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,154,0.0,0.0,4,0.0,0,0,0.0,154.0,0.0,0.0,2016-08-13 11:30:00+00:00,13 Aug 12:30,0.0,0.0,0,0.0,0.0,8,0,0.0,0,0,0.0,0,1,0,5792,0.0,0.0,0.0,1.0,2.0,0.0,0,0,0,0,40,True,0.0,0,1,2016-17,NaN,Hull,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,Leicester,3.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,4.0
4,Allan_McGregor,0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,146,0.0,0.0,4,0.0,0,0,0.0,146.0,0.0,0.0,2016-08-13 11:30:00+00:00,13 Aug 12:30,0.0,0.0,0,0.0,0.0,8,0,0.0,0,0,0.0,0,1,0,1141,0.0,0.0,0.0,1.0,2.0,0.0,0,0,0,0,45,True,0.0,0,1,2016-17,NaN,Hull,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,Leicester,3.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,4.5


## 7. Valider og lagre

Filen lagres med et nytt navn. Originalen `cleaned_merged_seasons_team_aggregated.csv` blir ikke overskrevet.

In [8]:
season_report = (
    expanded.groupby("season_x")
    .agg(rows=("name", "size"), players=("name", "nunique"), gameweeks=("GW", "nunique"))
    .reset_index()
)
display(season_report)

duplicate_keys = expanded.duplicated(
    ["season_x", "element", "fixture"], keep=False
).sum()
print("Duplikater på sesong + spiller + fixture:", duplicate_keys)
print("Andel med lagstatistikk:", expanded["points"].notna().mean().round(3))

OUTPUT_PATH = DATA_DIR / "cleaned_merged_seasons_team_aggregated_expanded.csv"
expanded.to_csv(OUTPUT_PATH, index=False)
print(f"Lagret {len(expanded):,} rader til:\n{OUTPUT_PATH}")

,season_x,rows,players,gameweeks
0,2016-17,23679,683,38
1,2017-18,22467,647,38
2,2018-19,21790,625,38
3,2019-20,22501,666,38
4,2020-21,24365,712,38
5,2021-22,25447,735,38
6,2022-23,26505,777,37
7,2023-24,29725,869,38
8,2024-25,27605,805,38
9,2025-26,29747,841,38


Duplikater på sesong + spiller + fixture: 0
Andel med lagstatistikk: 1.0
Lagret 254,441 rader til:
/Users/henrik/Documents/fplmodell/data-source/data/cleaned_merged_seasons_team_aggregated_expanded.csv
